# V6 single-stat effectiveness by year

Same definitions as **`v6_stat_correlations`** (bullpen gap, neutral pen when missing) — **without** correlation tables.

- Discovers every `games_<year>.parquet` under `data/`.
- For each season: completed games with `home_win`, then **favorite win rate** per single-stat rule.
- Output: long table + **pivot** (rule × year) + YoY deltas.

## Threshold behavior (`MIN_EDGES` in the next cell)

- **`min_edge = 0` (default)** — legacy: any non-null diff picks a side at zero (`>= 0` favors home for higher-better stats; xFIP and bullpen use `<= 0` for home).
- **`min_edge > 0`** — only games with a clear edge: for higher-better stats, home if diff `>= min_edge`, away if diff `<= -min_edge`; for xFIP and bullpen (lower home is better), home if diff `<= -min_edge`, away if diff `>= min_edge`. Toss-up band excluded from **n**.
- **Bullpen** — `pen_gap_fip` = home pen FIP − away pen FIP (lower is better for home; same sign story as `sp_xfip_diff`); thresholds apply **after missing → 0** (V6 neutral pen).

**Baseline:** `mean(home_win)` ≈ always picking home. **Coin flip** = 0.50.

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

FINAL_STATES = {"Final", "Game Over", "Completed Early"}

cwd = Path.cwd()
DATA = cwd / "data" if (cwd / "data").is_dir() else cwd.parent / "data"

# --- Tune: per-stat minimum |edge| before declaring a favorite ---
MIN_EDGE_DEFAULT = 0.0
MIN_EDGES = {
    "sp_kbb_diff": 0.0,
    "sp_xfip_diff": 0.0,
    "offense_diff": 0.0,
    "offense_platoon_diff": 0.0,
    "pen_gap_fip": 0.0,
}


def discover_season_parquets(data_dir: Path):
    out = []
    for p in sorted(data_dir.glob("games_*.parquet")):
        try:
            year = int(p.stem.split("_")[-1])
        except ValueError:
            continue
        out.append((year, p))
    return out


pairs = discover_season_parquets(DATA)
if not pairs:
    raise FileNotFoundError(f"No games_*.parquet under {DATA.resolve()}")

print("Data dir:", DATA.resolve())
print("Season files:", len(pairs), "—", [y for y, _ in pairs])
print("MIN_EDGES:", {k: MIN_EDGES.get(k, MIN_EDGE_DEFAULT) for k in MIN_EDGES})

In [ ]:
# V6-aligned prep: pen gap + thresholded favwon_* columns.

PEN_ABS = 0.75

STAT_SPECS = [
    ("sp_kbb_diff", "favwon_sp_kbb", "SP K-BB", True),
    ("sp_xfip_diff", "favwon_sp_xfip", "SP xFIP", False),
    ("offense_diff", "favwon_off", "Offense (wRC+)", True),
    ("offense_platoon_diff", "favwon_platoon", "Offense platoon", True),
    ("pen_gap_fip", "favwon_pen", "Bullpen (FIP gap)", False),
]


def preferred_pen_cols(frame: pd.DataFrame):
    if "home_pen_roll14_fip" in frame.columns:
        home_pen = frame["home_pen_roll14_fip"]
        if "home_pen_season_fip" in frame.columns:
            home_pen = home_pen.combine_first(frame["home_pen_season_fip"])
    elif "home_pen_season_fip" in frame.columns:
        home_pen = frame["home_pen_season_fip"]
    else:
        home_pen = pd.Series(np.nan, index=frame.index)

    if "away_pen_roll14_fip" in frame.columns:
        away_pen = frame["away_pen_roll14_fip"]
        if "away_pen_season_fip" in frame.columns:
            away_pen = away_pen.combine_first(frame["away_pen_season_fip"])
    elif "away_pen_season_fip" in frame.columns:
        away_pen = frame["away_pen_season_fip"]
    else:
        away_pen = pd.Series(np.nan, index=frame.index)

    return home_pen, away_pen


def favorite_won_thresholded(
    feat: pd.Series,
    min_edge: float,
    *,
    higher_is_better_for_home: bool,
    home_win: pd.Series,
) -> pd.Series:
    feat = pd.to_numeric(feat, errors="coerce")
    y = pd.to_numeric(home_win, errors="coerce")
    out = pd.Series(np.nan, index=feat.index, dtype="float")
    m = float(min_edge)

    if np.isclose(m, 0.0):
        if higher_is_better_for_home:
            prefer_home = feat >= 0
        else:
            prefer_home = feat <= 0
        valid = feat.notna() & y.notna()
        fav = np.where(prefer_home & valid, y == 1.0, np.where(valid, y == 0.0, np.nan))
        out[:] = fav
        return out

    if higher_is_better_for_home:
        home_pick = feat >= m
        away_pick = feat <= -m
    else:
        home_pick = feat <= -m
        away_pick = feat >= m

    valid = (home_pick | away_pick) & ~(home_pick & away_pick)
    valid = valid & feat.notna() & y.notna()
    prefer_home = home_pick & valid
    fav = np.where(prefer_home, y == 1.0, np.where(away_pick & valid, y == 0.0, np.nan))
    out[:] = fav
    return out


def add_favwon_columns(
    work: pd.DataFrame,
    *,
    min_edges: dict | None = None,
    min_edge_default: float = 0.0,
):
    me = min_edges or {}
    w = work.copy()
    home_pen, away_pen = preferred_pen_cols(w)
    w["pen_gap_fip"] = home_pen - away_pen
    w["pen_norm_v6"] = (w["pen_gap_fip"] / PEN_ABS).fillna(0.0)
    hw = w["home_win"]

    for raw_col, fav_col, _label, higher in STAT_SPECS:
        m = float(me.get(raw_col, min_edge_default))
        if raw_col == "pen_gap_fip":
            feat = pd.to_numeric(w["pen_gap_fip"], errors="coerce").fillna(0.0)
            w[fav_col] = favorite_won_thresholded(feat, m, higher_is_better_for_home=higher, home_win=hw)
        elif raw_col in w.columns:
            w[fav_col] = favorite_won_thresholded(w[raw_col], m, higher_is_better_for_home=higher, home_win=hw)
        else:
            w[fav_col] = np.nan

    return w


def effectiveness_rows(w: pd.DataFrame, season: int, *, min_edges: dict, min_edge_default: float):
    hw = pd.to_numeric(w["home_win"], errors="coerce")
    baseline = float(hw.mean()) if len(hw.dropna()) else float("nan")
    rows = []
    for raw_col, fav_col, label, _h in STAT_SPECS:
        if fav_col not in w.columns:
            continue
        m = float(min_edges.get(raw_col, min_edge_default))
        s = pd.to_numeric(w[fav_col], errors="coerce").dropna()
        n = int(len(s))
        if n == 0:
            rows.append(
                {
                    "season": season,
                    "rule": label,
                    "raw_col": raw_col,
                    "favwon_col": fav_col,
                    "min_edge": m,
                    "n": 0,
                    "favorite_win_rate": np.nan,
                    "se": np.nan,
                    "baseline_home_win_rate": baseline,
                }
            )
            continue
        p = float(s.mean())
        se = float(np.sqrt(p * (1.0 - p) / n))
        rows.append(
            {
                "season": season,
                "rule": label,
                "raw_col": raw_col,
                "favwon_col": fav_col,
                "min_edge": m,
                "n": n,
                "favorite_win_rate": p,
                "se": se,
                "baseline_home_win_rate": baseline,
            }
        )
    return rows

In [ ]:
# Load every season, compute effectiveness (long format).

all_rows = []
per_season_n_completed = {}

for season, parquet_path in pairs:
    df = pd.read_parquet(parquet_path).copy()
    df["game_date"] = pd.to_datetime(df["game_date"], errors="coerce").dt.normalize()
    df["detailed_state"] = df["detailed_state"].astype(str)
    work = df[df["detailed_state"].isin(FINAL_STATES)].copy()
    work = work[work["home_win"].notna()].copy()
    per_season_n_completed[season] = len(work)
    if len(work) == 0:
        print(f"{season}: 0 completed rows — skip")
        continue
    work = add_favwon_columns(work, min_edges=MIN_EDGES, min_edge_default=MIN_EDGE_DEFAULT)
    all_rows.extend(effectiveness_rows(work, season, min_edges=MIN_EDGES, min_edge_default=MIN_EDGE_DEFAULT))
    print(f"{season}: n_completed={len(work)}  file={parquet_path.name}")

effect_by_year = pd.DataFrame(all_rows).sort_values(["season", "rule"]).reset_index(drop=True)
display(effect_by_year)

print("\nCompleted games per season (input row counts):")
display(pd.Series(per_season_n_completed, name="n_completed").sort_index())

In [ ]:
# Year-over-year pivots

if effect_by_year.empty:
    print("No effectiveness rows; check parquet files.")
else:
    rate_pivot = effect_by_year.pivot(index="rule", columns="season", values="favorite_win_rate")
    n_pivot = effect_by_year.pivot(index="rule", columns="season", values="n")
    base_pivot = effect_by_year.drop_duplicates("season").set_index("season")["baseline_home_win_rate"]

    print("Favorite win rate (rule × season)")
    display(rate_pivot)

    print("\nSample n per rule × season")
    display(n_pivot)

    print("\nBaseline mean(home_win) per season")
    display(base_pivot.to_frame("baseline_home_win_rate"))

    rate_sorted = rate_pivot.reindex(sorted(rate_pivot.columns), axis=1)
    yoy = rate_sorted.diff(axis=1)
    print("\nYoY change in favorite_win_rate")
    display(yoy)